#### 문제
- GridSearchCV와 연동하기 위해서 Word2Vec class 생성한 것과 같이 해당 class 보강
- 모델을 생성할 수 있도록 생성자 함수 추가적인 작업
    - 3개의 매개변수 추가
        - min_n(기본값 : 2), max_n(기본값 : 4), bucket(기본값 : 2e+6)
    - 마지막 매개변수 1개 추가 
        - model을 선택할 수 있는 매개변수
        - model의 기본값은  'w2v'
- fit함수 수정
    - self.type에 따라서 학습이 되는 모델을 변경
        - 'w2v' 라면 Word2Vec 학습하고 self.model에 대입
        - 'ft'라면 FastText 학습하고 self.model에 대입

- 해당 클래스 모듈화
    - 모듈의 이름은 'gensim_test'.py

1. 모듈을 로드
2. tokenizer는 Okt 사용
    - okt.morphs()를 이용하여 토큰화
3. 데이터 셋은 ratings_test.txt 파일을 로드
4. 결측치 제거
5. 글자 간의 좌우 공백을 제거
6. 빈 테스트 데이터가 document에 존재하는가? 제외
7. 중복되는 document를 제외
8. 상위 데이터 100개를 이용하여 gridsearch를 이용해서 파라미터 조합
    - 파라미터 조합 (벡터화 : min_count=1로 고정)
        - type : ['w2v', 'ft']
        - vector_size : [80, 100]
    - 파라미터 조합 (학습 모델 : SVC)
        - C : [0.8, 1.0]
    - 계층화 폴드는 5회
9. 하위 데이터 100개를 이용해서 검증 : 분류 레포트(classification_report) 이용

In [6]:
#1. 
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import classification_report
from sklearn.svm import SVC
from konlpy.tag import Okt
import pandas as pd
# 커스텀 모듈에서 class만 로드
from gensim_test import Vectorizer

In [7]:
#3. 
df = pd.read_csv("./ratings_test.txt", sep='\t')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   id        50000 non-null  int64
 1   document  49997 non-null  str  
 2   label     50000 non-null  int64
dtypes: int64(2), str(1)
memory usage: 1.1 MB


In [8]:
#4.
#결측치 제외 
df.dropna(inplace=True)

#5. 
# document에서 좌우의 공백을 제거 
df['document'] = df['document'].str.strip()

#6. 
#빈 텍스트 존재하는가?
df.loc[df['document'] == '', ]

,id,document,label


In [9]:
# 빈 텍스트 제외
df = df.loc[~(df['document'] == ''), ]

In [10]:
#7. 
# document의 중복 데이터를 제거
df.drop_duplicates('document', inplace = True)
df.info()

<class 'pandas.DataFrame'>
Index: 49157 entries, 0 to 49999
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   id        49157 non-null  int64
 1   document  49157 non-null  str  
 2   label     49157 non-null  int64
dtypes: int64(2), str(1)
memory usage: 1.5 MB


In [11]:
#2. 
# 토큰화 함수 생성 
okt = Okt()

tokenizer = lambda x : [ word for word in okt.morphs(x) ]

In [12]:
pipe = Pipeline(
    [
        ('vector', Vectorizer(tokenizer=tokenizer, min_count=1)), 
        ('svc', SVC(random_state=42))
    ]
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [13]:
#8.
params = {
    'vector__type' : ['w2v', 'ft'], 
    'vector__vector_size' : [80, 100], 
    'vector__l2' : [False, True],
    'svc__C' : [0.8, 1.0]
}

grid = GridSearchCV(
    estimator= pipe, 
    param_grid= params, 
    cv = cv, 
    verbose=1
)


In [ ]:
X_train = df.head(100)['document'].values
y_train = df.head(100)['label'].values
X_test = df.tail(100)['document'].values
y_test = df.tail(100)['label'].values

grid.fit(X_train, y_train)

print("최적의 모델의 성능 점수 : ", grid.best_score_)

In [15]:
#9. 
pred = grid.predict(X_test)

print(classification_report(pred, y_test))

              precision    recall  f1-score   support

           0       0.78      0.60      0.68        67
           1       0.45      0.67      0.54        33

    accuracy                           0.62       100
   macro avg       0.62      0.63      0.61       100
weighted avg       0.67      0.62      0.63       100

